# Evaluación ergonómica RULA semiautomática

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MiguelSerranoReyes/ergonomia-digital/blob/main/rula_semiautomatico.ipynb)

Esta libreta calcula el puntaje **RULA** a partir de fotografías: la computadora detecta el cuerpo,
mide los ángulos de cada segmento y tú completas lo que una foto no puede decir. Al final obtienes el
puntaje de riesgo de cada postura.

Analizaremos tres casos:

| Caso | Situación |
|---|---|
| 1 | Oficinista frente a su computadora de escritorio |
| 2 | Persona trabajando con una laptop sobre las piernas |
| 3 | Odontóloga atendiendo a un paciente |

## ¿Qué es RULA?

**RULA** (*Rapid Upper Limb Assessment*, McAtamney y Corlett, 1993) es un método rápido para estimar
el riesgo de lesiones musculoesqueléticas por la **postura** durante el trabajo. Se enfoca en la parte
superior del cuerpo y se llena con una **hoja de evaluación** de 15 pasos.

La idea es sencilla: **cuanto más se aleja un segmento del cuerpo de su posición neutra, más puntos
suma.** Un brazo pegado al cuerpo suma poco; un brazo levantado por encima del hombro suma mucho.

La hoja se divide en dos grupos:

| Grupo | Segmentos | Pasos |
|---|---|---|
| **A. Brazo y muñeca** | brazo, antebrazo, muñeca, torcedura de muñeca | 1 a 8 |
| **B. Cuello, torso y piernas** | cuello, torso, piernas | 9 a 15 |

En cada grupo se sigue el mismo camino:

1. **Postura de cada segmento.** Se elige el rango de ángulo en el que está y se le asignan puntos.
2. **Ajustes.** Se suman o restan puntos por situaciones especiales: hombro levantado, cuello
   girado, brazo apoyado…
3. **Tabla.** Los puntajes de los segmentos se combinan en una tabla (Tabla A o Tabla B).
4. **Actividad.** Se suman puntos si la postura se sostiene mucho tiempo o se repite (uso muscular)
   y según el peso que se carga (fuerza/carga).

Por último, la **Tabla C** combina ambos grupos en el **puntaje RULA**, de 1 a 7:

| Puntaje RULA | Significado |
|---|---|
| 1 o 2 | Postura aceptable |
| 3 o 4 | Investigar más; podría requerir modificación |
| 5 o 6 | Investigar más y modificar pronto |
| 7 | Investigar e implementar modificaciones |

En esta libreta cada segmento tiene **siempre el mismo color**, en las fotos, en las preguntas y en
los resultados: brazo azul, antebrazo verde, muñeca gris, torcedura rosa, cuello rojo, torso naranja,
piernas amarillo y puntaje final morado.

## ¿Cómo se usa esta libreta?

Una **libreta** combina texto (como este) con **celdas de código** que la computadora ejecuta.

- Para ejecutar una celda, haz clic en ella y presiona **Shift + Enter** (o el botón ▶ a su izquierda).
- Ejecuta las celdas **en orden, de arriba hacia abajo**: cada una usa lo que preparó la anterior.
- Si algo falla o reinicias Colab, vuelve a ejecutar desde el principio
  (*Entorno de ejecución → Ejecutar todas*).
- Las líneas que empiezan con `#` son **comentarios**: notas para las personas que la computadora
  ignora.

No necesitas saber programar. En los recuadros 💡 explicamos lo básico de Python conforme aparece.

## Parte 0 · Preparar el entorno

### Instalar y cargar herramientas
Python trae pocas funciones de fábrica; el resto viene en **bibliotecas** que otras personas
escribieron. Primero instalamos **MediaPipe**, la biblioteca de Google que detecta el cuerpo en una
imagen.

💡 El signo `!` al inicio indica que es una instrucción para el sistema, no código de Python.

In [ ]:
# Instalar MediaPipe (tarda ~1 minuto)
!pip -q install mediapipe

Ahora **importamos** las bibliotecas: le decimos a Python que las vamos a usar.

💡 `import numpy as np` significa "usa la biblioteca numpy y llámala `np` para abreviar".
`resultados = {}` crea un **diccionario** vacío: una caja con etiquetas donde guardaremos todo lo
que calculemos de cada caso.

In [ ]:
import os, urllib.request
import numpy as np                   # cálculos con números y tablas
import pandas as pd                  # tablas de datos
import matplotlib.pyplot as plt      # gráficas e imágenes
import ipywidgets as widgets         # botones y controles interactivos
from IPython.display import display, HTML
from PIL import Image                # abrir imágenes
import mediapipe as mp               # detección del cuerpo

resultados = {}   # aquí se guarda todo lo que hagamos con cada caso
print("✓ Listo · MediaPipe", mp.__version__)

### Descargar las imágenes
Cada imagen trae **dos fotografías de la misma postura**: a la izquierda la vista lateral y a la
derecha la vista posterior. RULA necesita ambas: de lado se ve qué tanto se dobla el cuerpo hacia
adelante o hacia atrás, y desde atrás se ven las inclinaciones laterales y qué tanto se separan los
brazos.

💡 `CASOS` es un diccionario: a cada **archivo** le asocia un **nombre**. El `for` es un **ciclo**:
repite las instrucciones de abajo una vez por cada archivo.

In [ ]:
URL = "https://raw.githubusercontent.com/MiguelSerranoReyes/datasets/main/ergonomia/"
CASOS = {
    "01_oficina.png": "Oficinista",
    "02_laptop.png": "Trabajo con laptop",
    "03_odontologia.png": "Odontóloga",
}

os.makedirs("ergonomia", exist_ok=True)
for archivo in CASOS:
    ruta = f"ergonomia/{archivo}"
    if not os.path.exists(ruta):              # solo descarga si no la tenemos
        urllib.request.urlretrieve(URL + archivo, ruta)
    print("✓", archivo)

### Separar las dos vistas
Para la computadora, una imagen es una **tabla de números**: cada píxel tiene tres valores (rojo,
verde y azul) entre 0 y 255. Un píxel blanco es (255, 255, 255).

Entre las dos fotografías hay una **franja blanca**. Para encontrarla, la computadora recorre las
columnas de píxeles del centro y busca las que son muy brillantes y casi no cambian de arriba a
abajo. Ahí corta.

💡 `def` crea una **función**: un bloque de instrucciones con nombre que podemos reutilizar.
`cortar_vistas` recibe una imagen y **devuelve** (`return`) las dos vistas.

In [ ]:
def cortar_vistas(img):
    gris = img.mean(axis=2)                      # brillo de cada píxel (0 = negro, 255 = blanco)
    brillo = gris.mean(axis=0)                   # brillo promedio de cada columna
    variacion = gris.std(axis=0)                 # qué tanto cambia cada columna
    ancho = img.shape[1]
    centro = range(int(0.35 * ancho), int(0.65 * ancho))
    franja = [x for x in centro if brillo[x] > 235 and variacion[x] < 12]
    if not franja:
        raise RuntimeError("No encontré la franja blanca")
    return img[:, :franja[0]], img[:, franja[-1] + 1:]


vistas = {}
for archivo in CASOS:
    img = np.array(Image.open(f"ergonomia/{archivo}").convert("RGB"))
    lateral, posterior = cortar_vistas(img)
    vistas[archivo] = {"lateral": lateral, "posterior": posterior}
    print(f"✓ {archivo}: lateral {lateral.shape[1]} px · posterior {posterior.shape[1]} px de ancho")

Veamos las seis vistas ya separadas:

In [ ]:
fig, ejes = plt.subplots(3, 2, figsize=(12, 13))
for fila, (archivo, nombre) in zip(ejes, CASOS.items()):
    for eje, vista in zip(fila, ["lateral", "posterior"]):
        eje.imshow(vistas[archivo][vista])
        eje.set_title(f"{nombre} · {vista}")
        eje.axis("off")
plt.tight_layout()
plt.show()

### El detector de postura
**MediaPipe Pose** es una **red neuronal**: un modelo que aprendió, a partir de miles de fotografías
de personas, a reconocer dónde están las partes del cuerpo. Recibe una imagen y devuelve **33 puntos**
(orejas, hombros, codos, muñecas, caderas…). Para cada punto entrega:

- sus **coordenadas** en la imagen, y
- su **visibilidad**, entre 0 y 1: qué tan seguro está el modelo de haberlo visto. Un punto tapado
  por una silla tendrá visibilidad baja, aunque el modelo igual "adivina" dónde está.

Con estos puntos formamos segmentos (hombro→codo, cadera→hombro…) y medimos el ángulo entre ellos.
Esta celda descarga el modelo y lo prepara.

In [ ]:
MODELO = "pose_landmarker_full.task"
if not os.path.exists(MODELO):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_full/float16/1/pose_landmarker_full.task", MODELO)

vision = mp.tasks.vision
opciones = vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODELO),
    running_mode=vision.RunningMode.IMAGE,    # analizamos fotos, no video
    num_poses=1,                              # una persona por foto
)
detector = vision.PoseLandmarker.create_from_options(opciones)
print("✓ Detector de postura listo")

### Criterios de decisión
La hoja RULA no fija algunos umbrales. Por ejemplo, no dice desde cuántos grados un brazo cuenta como
"separado del cuerpo". Aquí los dejamos escritos para que sean visibles. **No son parte del método:
son decisiones nuestras**, y puedes cambiarlas.

💡 Para cambiar un criterio, modifica el número, ejecuta esta celda y después vuelve a ejecutar las
celdas de análisis del caso.

In [ ]:
CRITERIOS = {
    "muneca_neutra": 5,        # muñeca a menos de 5° se considera neutra (0°)
    "torso_neutro": 5,         # torso a menos de 5° de flexión cuenta como erguido
    "extension_cuello": 5,     # cuello más de 5° hacia atrás cuenta como extensión
    "hombro_elevado": 5,       # desnivel entre hombros mayor de 5° → hombro levantado
    "abduccion": 20,           # brazo separado más de 20° del torso → abducción
    "inclinacion_lateral": 10, # cuello o torso ladeados más de 10° → inclinación lateral
    "margen_costado": 0.25,    # muñeca más afuera que el hombro (25 % del ancho de hombros)
}

### Caja de herramientas 🔧
Las siguientes celdas contienen funciones ya escritas: medir ángulos, dibujar el esqueleto, las
tablas A, B y C, el cuestionario interactivo y los resúmenes. Están plegadas para no distraer;
**solo ejecútalas**. Si te da curiosidad, haz doble clic sobre ellas para ver el código.

La celda de las tablas vale la pena abrirla: son las mismas tablas de la hoja RULA, escritas como
listas de números.

In [ ]:
# @title 🔧 Herramientas de medición (ejecuta sin modificar) { display-mode: "form" }
# Funciones auxiliares: detectar puntos, medir ángulos y dibujar.
# No es necesario leer este código para usar la libreta.

PUNTOS = {
    "nariz": 0, "oreja_i": 7, "oreja_d": 8,
    "hombro_i": 11, "hombro_d": 12, "codo_i": 13, "codo_d": 14,
    "muneca_i": 15, "muneca_d": 16, "indice_i": 19, "indice_d": 20,
    "cadera_i": 23, "cadera_d": 24, "rodilla_i": 25, "rodilla_d": 26,
    "tobillo_i": 27, "tobillo_d": 28,
}
ARRIBA, ABAJO = np.array([0.0, -1.0]), np.array([0.0, 1.0])   # en imagen, y crece hacia abajo

# Colores de la hoja RULA: COLOR para líneas y bordes, TINTE para las casillas
COLOR = {"brazo": "#2f7fe0", "antebrazo": "#22a45d", "muneca": "#6c7a89", "torcedura": "#d6338a",
         "cuello": "#d93636", "torso": "#ee8a2a", "piernas": "#d4a90f",
         "muneca_brazo": "#7b5cc4", "cuello_torso_piernas": "#1fa3a8", "rula": "#5b47ff"}
TINTE = {"brazo": "#dbeafb", "antebrazo": "#d9f3e3", "muneca": "#dfe3e8", "torcedura": "#f9dcec",
         "cuello": "#f8d6d6", "torso": "#fde6cf", "piernas": "#fbf1c7",
         "muneca_brazo": "#e6dff5", "cuello_torso_piernas": "#d2f0f1", "rula": "#5b47ff"}


def detectar_puntos(img):
    """Corre MediaPipe y devuelve coordenadas (px) y visibilidad (0–1) de cada punto."""
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                   data=np.ascontiguousarray(img)))
    if not res.pose_landmarks:
        raise RuntimeError("MediaPipe no encontró a ninguna persona en la imagen.")
    lm = res.pose_landmarks[0]
    h, w = img.shape[:2]
    pts = {n: np.array([lm[i].x * w, lm[i].y * h]) for n, i in PUNTOS.items()}
    vis = {n: float(lm[i].visibility) for n, i in PUNTOS.items()}
    return pts, vis


def angulo_entre(v1, v2):
    """Ángulo sin signo entre dos vectores (0–180°)."""
    c = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


def angulo_con_signo(v, referencia, adelante):
    """Ángulo de v respecto a 'referencia'; positivo si v se inclina hacia 'adelante'."""
    return float(np.degrees(np.arctan2(np.dot(v, adelante), np.dot(v, referencia))))


def confiable(vis, nombres, umbral=0.5):
    return all(vis[n] >= umbral for n in nombres)


def etiqueta_conf(ok):
    return "alta" if ok else "BAJA ⚠"


# ------------------------------------------------------------------ vista lateral
def medir_lateral(img, lado=None):
    pts, vis = detectar_puntos(img)
    if lado is None:   # el lado más cercano a la cámara es el más visible
        prom = {s: np.mean([vis[f"{p}_{s}"] for p in ("hombro", "codo", "muneca", "cadera")])
                for s in ("i", "d")}
        lado = max(prom, key=prom.get)
    P = {p: pts[f"{p}_{lado}"] for p in ("oreja", "hombro", "codo", "muneca", "indice", "cadera")}
    V = {p: vis[f"{p}_{lado}"] for p in P}

    # ¿Hacia dónde mira la persona? La nariz queda delante de la oreja.
    signo = 1.0 if pts["nariz"][0] > P["oreja"][0] else -1.0
    ADELANTE = np.array([signo, 0.0])

    torso = angulo_con_signo(P["hombro"] - P["cadera"], ARRIBA, ADELANTE)
    cuello_vertical = angulo_con_signo(P["oreja"] - P["hombro"], ARRIBA, ADELANTE)
    cuello = cuello_vertical - torso                              # RULA: respecto al torso
    brazo = (angulo_con_signo(P["codo"] - P["hombro"], ABAJO, ADELANTE)
             - angulo_con_signo(P["cadera"] - P["hombro"], ABAJO, ADELANTE))
    codo = 180 - angulo_entre(P["hombro"] - P["codo"], P["muneca"] - P["codo"])
    muneca = angulo_entre(P["muneca"] - P["codo"], P["indice"] - P["muneca"])

    return {
        "img": img, "puntos": P, "lado": "izquierdo" if lado == "i" else "derecho",
        "mira_hacia": "derecha" if signo > 0 else "izquierda",
        "torso": torso, "cuello": cuello, "cuello_vertical": cuello_vertical,
        "brazo": brazo, "codo": codo, "muneca": muneca,
        "conf": {
            "torso": confiable(V, ["hombro", "cadera"]),
            "cuello": confiable(V, ["oreja", "hombro", "cadera"]),
            "brazo": confiable(V, ["hombro", "codo", "cadera"]),
            "codo": confiable(V, ["hombro", "codo", "muneca"]),
            "muneca": confiable(V, ["codo", "muneca", "indice"], 0.8),
        },
    }


# ---------------------------------------------------------------- vista posterior
def medir_posterior(img):
    pts, vis = detectar_puntos(img)
    # De espaldas, MediaPipe puede confundir izquierda y derecha:
    # ordenamos por posición en la imagen, no por la etiqueta del modelo.
    hi, hd = sorted([pts["hombro_i"], pts["hombro_d"]], key=lambda p: p[0])
    lados = sorted(["i", "d"], key=lambda s: pts[f"hombro_{s}"][0])   # [izq. imagen, der. imagen]
    c_hombros = (hi + hd) / 2
    c_orejas = (pts["oreja_i"] + pts["oreja_d"]) / 2
    c_caderas = (pts["cadera_i"] + pts["cadera_d"]) / 2
    ancho = np.linalg.norm(hd - hi)

    linea = hd - hi
    eje = np.array([linea[1], -linea[0]]) / (np.linalg.norm(linea) + 1e-9)   # perpendicular
    if eje[1] > 0:
        eje = -eje                                                         # que apunte hacia arriba

    inclin_hombros = float(np.degrees(np.arctan2(abs(linea[1]), abs(linea[0]))))
    cuello_lado = angulo_entre(c_orejas - c_hombros, eje)
    torso_lado = angulo_entre(c_hombros - c_caderas, ARRIBA)

    brazos, fuera_linea = {}, {}
    for s, nombre, hacia_fuera in [(lados[0], "brazo izq. imagen", -1), (lados[1], "brazo der. imagen", 1)]:
        v = pts[f"codo_{s}"] - pts[f"hombro_{s}"]
        brazos[nombre] = angulo_con_signo(v, -eje, np.array([hacia_fuera, 0.0]))
        x = pts[f"muneca_{s}"][0]
        cruza = (x - c_hombros[0]) * hacia_fuera < 0
        costado = (x - pts[f"hombro_{s}"][0]) * hacia_fuera > CRITERIOS["margen_costado"] * ancho
        fuera_linea[nombre] = cruza or costado

    return {
        "img": img, "c_hombros": c_hombros, "c_orejas": c_orejas, "c_caderas": c_caderas,
        "hombros": (hi, hd), "codos": (pts[f"codo_{lados[0]}"], pts[f"codo_{lados[1]}"]),
        "munecas": (pts[f"muneca_{lados[0]}"], pts[f"muneca_{lados[1]}"]),
        "inclin_hombros": inclin_hombros, "cuello_lado": cuello_lado, "torso_lado": torso_lado,
        "abduccion": brazos, "fuera_linea": fuera_linea,
        "conf": {
            "hombros": confiable(vis, ["hombro_i", "hombro_d"]),
            "cuello": confiable(vis, ["oreja_i", "oreja_d", "hombro_i", "hombro_d"]),
            "torso": confiable(vis, ["cadera_i", "cadera_d"], 0.8),
            "munecas": confiable(vis, ["muneca_i", "muneca_d"], 0.7),
        },
    }


# ---------------------------------------------------------------------- dibujo
def _seg(ax, a, b, color, estilo="-", g=3.5):
    ax.plot([a[0], b[0]], [a[1], b[1]], color=color, linestyle=estilo, linewidth=g,
            solid_capstyle="round")


def _etiqueta(ax, p, texto, segmento, dx=12, dy=-12):
    ax.annotate(texto, p, xytext=(dx, dy), textcoords="offset points", fontsize=9.5,
                color="#222", fontweight="bold",
                bbox=dict(facecolor=TINTE[segmento], edgecolor=COLOR[segmento], linewidth=2,
                          boxstyle="round,pad=0.3"))


def _punto(ax, p):
    ax.scatter(*p, color="white", edgecolor="black", s=45, zorder=5)


def dibujar_lateral(m, ax):
    P = m["puntos"]
    ax.imshow(m["img"])
    _seg(ax, P["cadera"], P["hombro"], COLOR["torso"])
    _seg(ax, P["hombro"], P["oreja"], COLOR["cuello"])
    _seg(ax, P["hombro"], P["codo"], COLOR["brazo"])
    _seg(ax, P["codo"], P["muneca"], COLOR["antebrazo"])
    _seg(ax, P["muneca"], P["indice"], COLOR["muneca"], g=2.5)
    for p in P.values():
        _punto(ax, p)
    _etiqueta(ax, P["oreja"], f"P9 Cuello {m['cuello']:+.0f}°", "cuello")
    _etiqueta(ax, (P["hombro"] + P["codo"]) / 2, f"P1 Brazo {m['brazo']:+.0f}°", "brazo", -110, 0)
    _etiqueta(ax, P["codo"], f"P2 Antebrazo {m['codo']:.0f}°", "antebrazo", -40, 30)
    _etiqueta(ax, P["indice"], f"P3 Muñeca ~{m['muneca']:.0f}°", "muneca", 14, -30)
    _etiqueta(ax, P["cadera"], f"P10 Torso {m['torso']:+.0f}°", "torso", -120, 10)
    ax.set_title(f"Vista lateral · lado {m['lado']}")
    ax.axis("off")


def dibujar_posterior(m, ax):
    ax.imshow(m["img"])
    hi, hd = m["hombros"]
    _seg(ax, hi, hd, COLOR["brazo"], g=2.5)
    _seg(ax, m["c_hombros"], m["c_orejas"], COLOR["cuello"])
    _seg(ax, m["c_hombros"], m["c_caderas"], COLOR["torso"], estilo="--")
    for h, c, w in zip(m["hombros"], m["codos"], m["munecas"]):
        _seg(ax, h, c, COLOR["brazo"])
        _seg(ax, c, w, COLOR["antebrazo"], g=2.5)
    for p in [m["c_orejas"], m["c_hombros"], m["c_caderas"], *m["hombros"], *m["codos"], *m["munecas"]]:
        _punto(ax, p)
    _etiqueta(ax, m["c_orejas"], f"P9a Cuello de lado {m['cuello_lado']:.0f}°", "cuello", 12, 10)
    _etiqueta(ax, hd, f"P1a Desnivel hombros {m['inclin_hombros']:.0f}°", "brazo", 12, -14)
    for c, (nombre, ang) in zip(m["codos"], m["abduccion"].items()):
        lado = -1 if "izq" in nombre else 1
        _etiqueta(ax, c, f"P1a Abducción {ang:.0f}°", "brazo", 12 if lado > 0 else -125, 0)
    _etiqueta(ax, m["c_caderas"], f"P10a Torso de lado {m['torso_lado']:.0f}°", "torso", 12, -10)
    ax.set_title("Vista posterior")
    ax.axis("off")


def analizar_caso(caso):
    """Mide ambas vistas, dibuja y guarda los resultados."""
    lat = medir_lateral(vistas[caso]["lateral"])
    post = medir_posterior(vistas[caso]["posterior"])
    fig, ejes = plt.subplots(1, 2, figsize=(15, 7))
    dibujar_lateral(lat, ejes[0])
    dibujar_posterior(post, ejes[1])
    plt.tight_layout()
    plt.show()
    resultados.setdefault(caso, {})["mediciones"] = {"lateral": lat, "posterior": post}
    return lat, post


def tabla_mediciones(lat, post):
    """Tabla con lo que midió la computadora, con la misma nomenclatura y colores de la hoja."""
    filas = [
        ("brazo", "Paso 1", "Posición de brazo (+ flexión, − extensión)", lat["brazo"], lat["conf"]["brazo"]),
        ("brazo", "Paso 1a", "Hombro levantado: desnivel entre hombros", post["inclin_hombros"], post["conf"]["hombros"]),
    ]
    for nombre, ang in post["abduccion"].items():
        filas.append(("brazo", "Paso 1a", f"Brazo en abducción ({nombre})", ang, post["conf"]["hombros"]))
    filas += [
        ("antebrazo", "Paso 2", "Posición de antebrazo (flexión del codo)", lat["codo"], lat["conf"]["codo"]),
        ("muneca", "Paso 3", "Posición de muñeca (aprox., con el dedo índice)", lat["muneca"], lat["conf"]["muneca"]),
        ("cuello", "Paso 9", "Posición del cuello respecto al torso (+ flexión, − extensión)", lat["cuello"], lat["conf"]["cuello"]),
        ("cuello", "Paso 9a", "Cuello se inclina de lado", post["cuello_lado"], post["conf"]["cuello"]),
        ("torso", "Paso 10", "Posición del torso (+ flexión, − reclinado)", lat["torso"], lat["conf"]["torso"]),
        ("torso", "Paso 10a", "Torso se inclina de lado", post["torso_lado"], post["conf"]["torso"]),
    ]
    html = ("<table style='border-collapse:collapse;font-size:14px;color:#222;background:#fff'>"
            "<tr style='background:#5f6b7a;color:white'><th style='padding:6px 10px'>Paso</th>"
            "<th style='padding:6px 10px;text-align:left'>Medición</th><th style='padding:6px 10px'>Ángulo</th>"
            "<th style='padding:6px 10px'>¿Confiable?</th></tr>")
    for seg, paso, nombre, ang, ok in filas:
        html += (f"<tr style='background:{TINTE[seg]};border-left:6px solid {COLOR[seg]}'>"
                 f"<td style='padding:4px 10px'><b>{paso}</b></td><td style='padding:4px 10px'>{nombre}</td>"
                 f"<td style='padding:4px 10px;text-align:right'>{ang:+.1f}°</td>"
                 f"<td style='padding:4px 10px'>{'✓' if ok else '⚠ revisar'}</td></tr>")
    display(HTML(html + "</table>"))

In [ ]:
# @title 📋 Tablas RULA A, B y C (las mismas de la hoja) { display-mode: "form" }

# Tabla A — filas: brazo (1–6) × antebrazo (1–3)
# columnas: muñeca 1–4, cada una con torcedura 1 y 2
TABLA_A = [
    [[1, 2, 2, 2, 2, 3, 3, 3], [2, 2, 2, 2, 3, 3, 3, 3], [2, 3, 3, 3, 3, 3, 4, 4]],  # brazo 1
    [[2, 3, 3, 3, 3, 4, 4, 4], [3, 3, 3, 3, 3, 4, 4, 4], [3, 4, 4, 4, 4, 4, 5, 5]],  # brazo 2
    [[3, 3, 4, 4, 4, 4, 5, 5], [3, 4, 4, 4, 4, 4, 5, 5], [4, 4, 4, 4, 4, 5, 5, 5]],  # brazo 3
    [[4, 4, 4, 4, 4, 5, 5, 5], [4, 4, 4, 4, 4, 5, 5, 5], [4, 4, 4, 5, 5, 5, 6, 6]],  # brazo 4
    [[5, 5, 5, 5, 5, 6, 6, 7], [5, 6, 6, 6, 6, 7, 7, 7], [6, 6, 6, 7, 7, 7, 7, 8]],  # brazo 5
    [[7, 7, 7, 7, 7, 8, 8, 9], [8, 8, 8, 8, 8, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9]],  # brazo 6
]

# Tabla B — filas: cuello (1–6); columnas: torso 1–6, cada uno con piernas 1 y 2
TABLA_B = [
    [1, 3, 2, 3, 3, 4, 5, 5, 6, 6, 7, 7],
    [2, 3, 2, 3, 4, 5, 5, 5, 6, 7, 7, 7],
    [3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7],
    [5, 5, 5, 6, 6, 7, 7, 7, 7, 7, 8, 8],
    [7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8],
    [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9],
]

# Tabla C — filas: puntaje muñeca y brazo (1…8+); columnas: cuello, torso y piernas (1…7+)
TABLA_C = [
    [1, 2, 3, 3, 4, 5, 5],
    [2, 2, 3, 4, 4, 5, 5],
    [3, 3, 3, 4, 4, 5, 6],
    [3, 3, 3, 4, 5, 6, 6],
    [4, 4, 4, 5, 6, 7, 7],
    [4, 4, 5, 6, 6, 7, 7],
    [5, 5, 6, 6, 7, 7, 7],
    [5, 5, 6, 7, 7, 7, 7],
]


def consultar_A(brazo, antebrazo, muneca, torcedura):
    return TABLA_A[brazo - 1][antebrazo - 1][(muneca - 1) * 2 + (torcedura - 1)]


def consultar_B(cuello, torso, piernas):
    return TABLA_B[cuello - 1][(torso - 1) * 2 + (piernas - 1)]


def consultar_C(fila, columna):
    return TABLA_C[min(fila, 8) - 1][min(columna, 7) - 1]


def interpretar(rula):
    if rula <= 2: return "Postura aceptable", "#2e7d32"
    if rula <= 4: return "Investigar más; podría requerir modificación", "#f9a825"
    if rula <= 6: return "Investigar más y modificar pronto", "#ef6c00"
    return "Investigar e implementar modificaciones", "#c62828"


# Comprobación rápida con el ejemplo de la hoja
assert consultar_A(1, 2, 1, 1) == 2 and consultar_B(2, 2, 1) == 2 and consultar_C(3, 3) == 3
print("✓ Tablas RULA cargadas")

In [ ]:
# @title 🧩 Cuestionario interactivo (ejecuta sin modificar) { display-mode: "form" }
# Tarjetas con los mismos pasos, textos y colores de la hoja RULA.
# Combina lo medido con lo que tú observas y calcula el puntaje en vivo.

NO = ("No", 0)
CARGA = [("Menos de 4.4 lb (2 kg), intermitente", 0),
         ("De 4.4 a 22 lb (2–10 kg), intermitente", 1),
         ("De 4.4 a 22 lb (2–10 kg), estática o repetida", 2),
         ("Mayor a 22 lb (10 kg), o repetida o con choque", 3)]
USO = [NO, ("Postura principalmente estática (> 1 min) o acciones que se repiten 4 veces por minuto", 1)]
DECIDE = "La computadora no puede saberlo con una foto: decide tú."


def _conf(ok):
    return "" if ok else " <b style='color:#c62828'>(medición poco confiable: revisa la imagen)</b>"


def _rango(valor, cortes):
    """Índice del primer corte que no supera 'valor'."""
    for i, c in enumerate(cortes):
        if valor <= c:
            return i
    return len(cortes)


def crear_preguntas(lat, post):
    C = CRITERIOS
    ab_max = max(post["abduccion"].values())
    return [
        # ------------------------------------------------- A. Análisis de brazo y muñeca
        dict(id="brazo", seg="brazo", paso="Paso 1", titulo="Posición de brazo",
             que="Ángulo del brazo (hombro→codo) respecto al torso, visto de lado. "
                 "Hacia adelante es flexión; hacia atrás, extensión.",
             como="En la foto lateral, compara la línea azul (brazo) con la naranja (torso).",
             opciones=[("20° en extensión a 20° de flexión (20–20°)", 1), ("Más de 20° en extensión", 2),
                       ("20–45°", 2), ("45–90°", 3), ("90°+", 4)],
             sug=(1 if lat["brazo"] < -20 else [0, 2, 3, 4][_rango(lat["brazo"], [20, 45, 90])],
                  f"Medí {lat['brazo']:+.0f}°." + _conf(lat["conf"]["brazo"]))),
        dict(id="hombro", seg="brazo", paso="Paso 1a", titulo="Ajuste: hombro levantado",
             que="El hombro sube hacia la oreja, por alcanzar algo alto o por tensión.",
             como="En la foto posterior, ¿la línea azul entre hombros está inclinada?",
             opciones=[NO, ("Sí, hombro levantado", 1)],
             sug=(int(post["inclin_hombros"] > C["hombro_elevado"]),
                  f"Desnivel entre hombros de {post['inclin_hombros']:.0f}° (criterio: más de "
                  f"{C['hombro_elevado']}°). Si ambos hombros suben por igual, la computadora no lo nota."
                  + _conf(post["conf"]["hombros"]))),
        dict(id="abduccion", seg="brazo", paso="Paso 1a", titulo="Ajuste: brazo en abducción",
             que="El brazo se abre hacia el costado, alejándose del torso.",
             como="En la foto posterior, ¿los codos se alejan hacia afuera?",
             opciones=[NO, ("Sí, brazo en abducción", 1)],
             sug=(int(ab_max > C["abduccion"]),
                  "Medí " + " y ".join(f"{v:.0f}° ({k})" for k, v in post["abduccion"].items())
                  + f" (criterio: más de {C['abduccion']}°).")),
        dict(id="apoyo", seg="brazo", paso="Paso 1a", titulo="Ajuste: brazo apoyado o persona recargada",
             que="Si el peso del brazo descansa en algo, o la persona se recarga, el hombro trabaja "
                 "menos y se resta 1.",
             como="Busca descansabrazos, el borde del escritorio o antebrazos sobre las piernas.",
             opciones=[NO, ("Sí, brazo apoyado o persona recargada", -1)], sug=(None, DECIDE)),
        dict(id="antebrazo", seg="antebrazo", paso="Paso 2", titulo="Posición de antebrazo",
             que="Cuánto se dobla el codo. Brazo estirado = 0°; codo en ángulo recto = 90°.",
             como="En la foto lateral, compara la línea verde (antebrazo) con la azul (brazo).",
             opciones=[("60–100°", 1), ("0–60°", 2), ("100°+", 2)],
             sug=(0 if 60 <= lat["codo"] <= 100 else 1 if lat["codo"] < 60 else 2,
                  f"Medí {lat['codo']:.0f}°." + _conf(lat["conf"]["codo"]))),
        dict(id="linea_media", seg="antebrazo", paso="Paso 2a",
             titulo="Ajuste: cualquier brazo pasa de la línea media o sale a los costados",
             que="La mano trabaja del otro lado del cuerpo, o muy por fuera del hombro.",
             como="En la foto posterior, imagina una línea vertical por el centro de la espalda.",
             opciones=[NO, ("Sí, pasa de la línea media o sale a los costados", 1)],
             sug=(int(any(post["fuera_linea"].values())),
                  "Según la posición de las muñecas: "
                  + ", ".join(f"{k}: {'sí' if v else 'no'}" for k, v in post["fuera_linea"].items()) + "."
                  + _conf(post["conf"]["munecas"]))),
        dict(id="muneca", seg="muneca", paso="Paso 3", titulo="Posición de muñeca",
             que="Cuánto se dobla la mano hacia arriba (extensión) o hacia abajo (flexión). "
                 "Mano alineada con el antebrazo = 0°.",
             como="En la foto lateral, compara la línea gris (mano) con la verde (antebrazo).",
             opciones=[("0°", 1), ("15–15° (hasta 15° hacia arriba o hacia abajo)", 2),
                       ("15°+ (flexión o extensión)", 3)],
             sug=(_rango(lat["muneca"], [C["muneca_neutra"], 15]),
                  f"Estimé ~{lat['muneca']:.0f}° con el dedo índice. Es la medición más débil: "
                  "la mano ocupa pocos píxeles." + _conf(lat["conf"]["muneca"]))),
        dict(id="desviacion", seg="muneca", paso="Paso 3a", titulo="Ajuste: muñeca desviada de línea media",
             que="La mano se inclina hacia el pulgar o hacia el meñique.",
             como="Se aprecia mejor desde arriba o desde atrás.",
             opciones=[NO, ("Sí, muñeca desviada", 1)], sug=(None, DECIDE)),
        dict(id="torcedura", seg="torcedura", paso="Paso 4", titulo="Torcedura de muñeca",
             que="Giro del antebrazo: de palma hacia abajo a palma hacia arriba.",
             como="Rango medio: giro cómodo. Final del rango: girado al tope, como al forzar un desarmador.",
             opciones=[("Muñeca torcida en rango medio", 1), ("Muñeca cerca o al final del rango", 2)],
             sug=(None, DECIDE)),
        dict(id="musculo_A", seg="muneca_brazo", paso="Paso 6", titulo="Sumar puntaje de uso muscular",
             que="Brazos y muñecas: postura sostenida o movimiento repetido.",
             como="Piensa en la tarea completa, no solo en el instante de la foto.",
             opciones=USO, sug=(None, "Depende de la tarea: decide tú.")),
        dict(id="fuerza_A", seg="muneca_brazo", paso="Paso 7", titulo="Sumar puntaje fuerza/carga",
             que="Peso que se sostiene o fuerza que se aplica con brazos y manos.",
             como="1 kg ≈ 2.2 lb.",
             opciones=CARGA, sug=(None, "Depende de la tarea: decide tú.")),
        # --------------------------------------------- B. Análisis de cuello, torso y piernas
        dict(id="cuello", seg="cuello", paso="Paso 9", titulo="Posición del cuello",
             que="Inclinación de la cabeza hacia adelante respecto al torso (no respecto a la vertical).",
             como="En la foto lateral, compara la línea roja (cuello) con la naranja (torso).",
             opciones=[("0–10°", 1), ("10–20°", 2), ("20°+", 3), ("En extensión", 4)],
             sug=(3 if lat["cuello"] < -C["extension_cuello"] else _rango(max(lat["cuello"], 0), [10, 20]),
                  f"Medí {lat['cuello']:+.0f}° respecto al torso ({lat['cuello_vertical']:+.0f}° respecto a la "
                  "vertical). La línea oreja-hombro no es la columna cervical: suele exagerar la flexión."
                  + _conf(lat["conf"]["cuello"]))),
        dict(id="cuello_torcido", seg="cuello", paso="Paso 9a", titulo="Ajuste: cuello torcido",
             que="La cabeza gira hacia un lado, como al voltear a otra pantalla.",
             como="En 2D es difícil de medir: obsérvalo en ambas fotos.",
             opciones=[NO, ("Sí, cuello torcido", 1)], sug=(None, DECIDE)),
        dict(id="cuello_lado", seg="cuello", paso="Paso 9a", titulo="Ajuste: cuello se inclina de lado",
             que="La cabeza se ladea hacia un hombro.",
             como="En la foto posterior, ¿la línea roja se aleja de la perpendicular a los hombros?",
             opciones=[NO, ("Sí, cuello inclinado de lado", 1)],
             sug=(int(post["cuello_lado"] > C["inclinacion_lateral"]),
                  f"Medí {post['cuello_lado']:.0f}° (criterio: más de {C['inclinacion_lateral']}°)."
                  + _conf(post["conf"]["cuello"]))),
        dict(id="torso", seg="torso", paso="Paso 10", titulo="Posición del torso",
             que="Inclinación del torso hacia adelante. Sentado y bien apoyado en el respaldo cuenta como 0°.",
             como="En la foto lateral, compara la línea naranja (torso) con la vertical.",
             opciones=[("0° (o sentado bien apoyado)", 1), ("0–20°", 2), ("20–60°", 3), ("60°+", 4)],
             sug=(0 if lat["torso"] <= C["torso_neutro"] else _rango(lat["torso"], [20, 60]) + 1,
                  f"Medí {lat['torso']:+.0f}°" + (" (reclinado hacia atrás)." if lat["torso"] < 0 else ".")
                  + _conf(lat["conf"]["torso"]))),
        dict(id="torso_torcido", seg="torso", paso="Paso 10a", titulo="Ajuste: torso torcido",
             que="Hombros y cadera no miran hacia el mismo lado.",
             como="En 2D es difícil de medir: obsérvalo en ambas fotos.",
             opciones=[NO, ("Sí, torso torcido", 1)], sug=(None, DECIDE)),
        dict(id="torso_lado", seg="torso", paso="Paso 10a", titulo="Ajuste: torso se inclina de lado",
             que="El torso se ladea hacia la derecha o la izquierda.",
             como="En la foto posterior, ¿la línea naranja punteada está vertical?",
             opciones=[NO, ("Sí, torso inclinado de lado", 1)],
             sug=(int(post["torso_lado"] > C["inclinacion_lateral"]),
                  f"Medí {post['torso_lado']:.0f}° (criterio: más de {C['inclinacion_lateral']}°). "
                  "Desde atrás la cadera suele quedar tapada por la silla." + _conf(post["conf"]["torso"]))),
        dict(id="piernas", seg="piernas", paso="Paso 11", titulo="Piernas",
             que="Ambos pies apoyados y el peso repartido de forma equilibrada.",
             como="¿Los pies tocan el piso o un reposapiés?",
             opciones=[("Piernas y pies con apoyo", 1), ("Sin apoyo", 2)], sug=(None, DECIDE)),
        dict(id="musculo_B", seg="cuello_torso_piernas", paso="Paso 13", titulo="Sumar puntaje de uso muscular",
             que="Cuello, torso y piernas: postura sostenida o movimiento repetido.",
             como="Piensa en la tarea completa.",
             opciones=USO, sug=(None, "Depende de la tarea: decide tú.")),
        dict(id="fuerza_B", seg="cuello_torso_piernas", paso="Paso 14", titulo="Sumar puntaje fuerza/carga",
             que="Peso o fuerza que recae sobre cuello, torso o piernas.",
             como="1 kg ≈ 2.2 lb.",
             opciones=CARGA, sug=(None, "Depende de la tarea: decide tú.")),
    ]


def calcular_rula(p):
    """p: puntos elegidos en cada pregunta. Devuelve todas las casillas de la hoja."""
    r = {}
    r["brazo"] = int(np.clip(p["brazo"] + p["hombro"] + p["abduccion"] + p["apoyo"], 1, 6))
    r["antebrazo"] = int(np.clip(p["antebrazo"] + p["linea_media"], 1, 3))
    r["muneca"] = int(np.clip(p["muneca"] + p["desviacion"], 1, 4))
    r["torcedura"] = p["torcedura"]
    r["postura_A"] = consultar_A(r["brazo"], r["antebrazo"], r["muneca"], r["torcedura"])
    r["musculo_A"], r["fuerza_A"] = p["musculo_A"], p["fuerza_A"]
    r["muneca_brazo"] = r["postura_A"] + r["musculo_A"] + r["fuerza_A"]
    r["cuello"] = int(np.clip(p["cuello"] + p["cuello_torcido"] + p["cuello_lado"], 1, 6))
    r["torso"] = int(np.clip(p["torso"] + p["torso_torcido"] + p["torso_lado"], 1, 6))
    r["piernas"] = p["piernas"]
    r["postura_B"] = consultar_B(r["cuello"], r["torso"], r["piernas"])
    r["musculo_B"], r["fuerza_B"] = p["musculo_B"], p["fuerza_B"]
    r["cuello_torso_piernas"] = r["postura_B"] + r["musculo_B"] + r["fuerza_B"]
    r["rula"] = consultar_C(r["muneca_brazo"], r["cuello_torso_piernas"])
    return r


# Casillas de la hoja, en su orden: (clave, nombre en la hoja, color)
CASILLAS_A = [("brazo", "Puntaje de brazo", "brazo"), ("antebrazo", "Puntaje de antebrazo", "antebrazo"),
              ("muneca", "Puntaje de muñeca", "muneca"),
              ("torcedura", "Puntaje de torcedura de muñeca", "torcedura"),
              ("postura_A", "Paso 5 · Puntaje postura A (Tabla A)", None),
              ("musculo_A", "+ Paso 6 · Puntaje de uso muscular", None),
              ("fuerza_A", "+ Paso 7 · Puntaje fuerza/carga", None),
              ("muneca_brazo", "= Paso 8 · Puntaje de muñeca y brazo", "muneca_brazo")]
CASILLAS_B = [("cuello", "Puntaje de cuello", "cuello"), ("torso", "Puntaje de torso", "torso"),
              ("piernas", "Puntaje de piernas", "piernas"),
              ("postura_B", "Paso 12 · Puntaje de postura B (Tabla B)", None),
              ("musculo_B", "+ Paso 13 · Puntaje de uso muscular", None),
              ("fuerza_B", "+ Paso 14 · Puntaje fuerza/carga", None),
              ("cuello_torso_piernas", "= Paso 15 · Puntaje de cuello, torso y piernas", "cuello_torso_piernas")]


def hoja_html(r, titulo):
    """Dibuja las casillas de la hoja RULA con los puntajes calculados."""
    def col(encabezado, casillas):
        filas = ""
        for clave, nombre, seg in casillas:
            fondo = TINTE[seg] if seg else "#ffffff"
            borde = COLOR[seg] if seg else "#bbb"
            filas += (f"<tr><td style='padding:5px 10px'>{nombre}</td>"
                      f"<td style='background:{fondo};border:2px solid {borde};text-align:center;"
                      f"font-weight:bold;font-size:16px;width:46px'>{r[clave]}</td></tr>")
        return (f"<table style='border-collapse:separate;border-spacing:0 4px'>"
                f"<tr><th colspan=2 style='background:#5f6b7a;color:white;padding:6px'>{encabezado}</th></tr>"
                f"{filas}</table>")
    txt, color = interpretar(r["rula"])
    return (f"<div style='background:#fff;color:#222;border:1px solid #ccc;border-radius:8px;padding:12px;"
            f"font-size:14px'><b>{titulo}</b>"
            f"<div style='display:flex;gap:30px;flex-wrap:wrap;margin-top:8px'>"
            f"{col('A. Análisis de brazo y muñeca', CASILLAS_A)}"
            f"{col('B. Análisis de cuello, torso y piernas', CASILLAS_B)}</div>"
            f"<div style='margin-top:10px;display:flex;gap:12px;align-items:center'>"
            f"<div style='background:{COLOR['rula']};color:white;font-size:22px;font-weight:bold;"
            f"padding:10px 18px;border-radius:6px'>Puntaje RULA: {r['rula']}</div>"
            f"<div>Tabla C (fila {min(r['muneca_brazo'], 8)}, columna {min(r['cuello_torso_piernas'], 7)})<br>"
            f"<b style='color:{color}'>{txt}</b></div></div></div>")


_TARJETA = ("background:#ffffff;color:#222;border:1px solid #ddd;border-left:8px solid {c};"
            "border-radius:6px;padding:10px 14px;margin:12px 0 2px 0;font-size:14px;line-height:1.45")


def _tarjeta(q):
    idx, texto = q["sug"]
    if idx is None:
        sug = f"🧑 {texto}"
    else:
        etiqueta, pts = q["opciones"][idx]
        sug = f"🤖 <b>Sugerencia:</b> {etiqueta} ({pts:+d}). {texto}"
    return widgets.HTML(
        f"<div style='{_TARJETA.format(c=COLOR[q['seg']])}'>"
        f"<span style='background:{TINTE[q['seg']]};padding:1px 6px;border-radius:4px'><b>{q['paso']}</b></span> "
        f"<b style='font-size:15px'>{q['titulo']}</b><br>"
        f"📖 {q['que']}<br>👀 <i>{q['como']}</i><br>{sug}</div>")


def cuestionario_rula(caso):
    lat = resultados[caso]["mediciones"]["lateral"]
    post = resultados[caso]["mediciones"]["posterior"]
    preguntas = crear_preguntas(lat, post)
    controles, avisos = {}, {}
    panel = widgets.HTML()

    def actualizar(_=None):
        elegidas = {q["id"]: q["opciones"][controles[q["id"]].value] for q in preguntas}
        for q in preguntas:
            idx = q["sug"][0]
            cambio = idx is not None and controles[q["id"]].value != idx
            avisos[q["id"]].value = ("<span style='color:#b26a00'>✏️ Cambiaste la sugerencia de la "
                                     "computadora. ¿Por qué?</span>" if cambio else "")
        r = calcular_rula({k: pts for k, (_, pts) in elegidas.items()})
        resultados[caso]["puntajes"] = r
        resultados[caso]["respuestas"] = [(q["paso"], q["titulo"], q["seg"], *elegidas[q["id"]])
                                          for q in preguntas]
        panel.value = hoja_html(r, "Puntaje en vivo (se actualiza al cambiar cualquier respuesta)")

    secciones = {"A": [], "B": []}
    for q in preguntas:
        idx = q["sug"][0]
        ctrl = widgets.RadioButtons(
            options=[(f"{t}   ({p:+d})", j) for j, (t, p) in enumerate(q["opciones"])],
            value=idx if idx is not None else 0, layout=widgets.Layout(width="auto"))
        ctrl.observe(actualizar, names="value")
        controles[q["id"]], avisos[q["id"]] = ctrl, widgets.HTML()
        grupo = "B" if q["seg"] in ("cuello", "torso", "piernas", "cuello_torso_piernas") else "A"
        secciones[grupo] += [_tarjeta(q), ctrl, avisos[q["id"]]]

    pestanas = widgets.Tab(children=[widgets.VBox(secciones["A"]), widgets.VBox(secciones["B"])])
    pestanas.set_title(0, "A. Brazo y muñeca (pasos 1–8)")
    pestanas.set_title(1, "B. Cuello, torso y piernas (pasos 9–15)")

    fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
    dibujar_lateral(lat, ejes[0])
    dibujar_posterior(post, ejes[1])
    plt.tight_layout()
    plt.show()
    actualizar()
    display(pestanas, panel)

In [ ]:
# @title 📝 Resúmenes de resultados (ejecuta sin modificar) { display-mode: "form" }

def resumen(caso):
    """Muestra, paso por paso, lo que se eligió y las casillas finales de la hoja."""
    datos = resultados.get(caso, {})
    if "puntajes" not in datos:
        print("⚠ Primero ejecuta el cuestionario de este caso.")
        return
    filas = ""
    for paso, titulo, seg, respuesta, pts in datos["respuestas"]:
        filas += (f"<tr style='background:{TINTE[seg]};border-left:6px solid {COLOR[seg]}'>"
                  f"<td style='padding:4px 10px'><b>{paso}</b></td><td style='padding:4px 10px'>{titulo}</td>"
                  f"<td style='padding:4px 10px'>{respuesta}</td>"
                  f"<td style='padding:4px 10px;text-align:right'><b>{pts:+d}</b></td></tr>")
    display(HTML(
        f"<h3 style='margin-bottom:4px'>{CASOS[caso]}: respuestas paso por paso</h3>"
        "<table style='border-collapse:collapse;font-size:14px;color:#222;background:#fff'>"
        "<tr style='background:#5f6b7a;color:white'><th style='padding:6px 10px'>Paso</th>"
        "<th style='padding:6px 10px;text-align:left'>Pregunta</th>"
        "<th style='padding:6px 10px;text-align:left'>Respuesta</th><th style='padding:6px 10px'>Puntos</th></tr>"
        f"{filas}</table><br>"))
    display(HTML(hoja_html(datos["puntajes"], "Hoja de evaluación RULA")))


def comparar_casos():
    """Pone lado a lado las casillas de todos los casos evaluados."""
    listos = [c for c in CASOS if "puntajes" in resultados.get(c, {})]
    if not listos:
        print("⚠ Aún no hay casos evaluados.")
        return
    filas = ""
    for clave, nombre, seg in CASILLAS_A + CASILLAS_B + [("rula", "Puntaje RULA", "rula")]:
        estilo = (f"background:{TINTE[seg]};border-left:6px solid {COLOR[seg]}" if seg and seg != "rula"
                  else f"background:{COLOR['rula']};color:white" if seg == "rula" else "background:#fff")
        celdas = "".join(f"<td style='padding:4px 14px;text-align:center'><b>{resultados[c]['puntajes'][clave]}</b></td>"
                         for c in listos)
        filas += f"<tr style='{estilo}'><td style='padding:4px 10px'>{nombre}</td>{celdas}</tr>"
    interpretaciones = "".join(
        f"<td style='padding:4px 10px;font-size:12px;color:{interpretar(resultados[c]['puntajes']['rula'])[1]}'>"
        f"<b>{interpretar(resultados[c]['puntajes']['rula'])[0]}</b></td>" for c in listos)
    encabezado = "".join(f"<th style='padding:6px 14px'>{CASOS[c]}</th>" for c in listos)
    display(HTML(
        "<table style='border-collapse:collapse;font-size:14px;color:#222;background:#fff'>"
        f"<tr style='background:#5f6b7a;color:white'><th style='padding:6px 10px;text-align:left'>Casilla</th>"
        f"{encabezado}</tr>{filas}<tr><td></td>{interpretaciones}</tr></table>"))

---
## Parte 1 · Caso 1: oficinista

### 1.1 La situación
La oficinista pasa varias horas al día escribiendo en el teclado y leyendo en el monitor.
Está sentada en una silla con respaldo y descansabrazos; no carga objetos.

💡 Guardamos el nombre del archivo en la **variable** `CASO`. Las celdas siguientes la usan, así
que es lo único que cambia de un caso a otro.

In [ ]:
CASO = "01_oficina.png"

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
for eje, vista in zip(ejes, ["lateral", "posterior"]):
    eje.imshow(vistas[CASO][vista])
    eje.set_title(f"{CASOS[CASO]} · vista {vista}")
    eje.axis("off")
plt.tight_layout()
plt.show()

### 1.2 Lo que mide la computadora 🤖
La función `analizar_caso` detecta los puntos del cuerpo en ambas vistas, mide los ángulos y los
dibuja. Cada etiqueta lleva el **número de paso** de la hoja RULA y su color:

- **Vista lateral:** P1 Brazo, P2 Antebrazo, P3 Muñeca (aproximada), P9 Cuello y P10 Torso.
- **Vista posterior:** los ajustes P1a (hombro levantado y abducción), P9a (cuello inclinado de
  lado) y P10a (torso inclinado de lado).

Los ángulos de la vista lateral tienen **signo**: positivo es hacia adelante (flexión) y negativo
hacia atrás (extensión o reclinado). Para saber qué es "adelante", la computadora revisa hacia
dónde apunta la nariz.

Dos detalles importantes:
- El **cuello** se mide respecto al **torso**, como pide RULA, no respecto a la vertical.
- Desde atrás, MediaPipe puede confundir izquierda y derecha (aprendió sobre todo con personas de
  frente), por eso nombramos los brazos según su lado **en la imagen**.

💡 `lat, post = analizar_caso(CASO)` guarda en dos variables lo que devuelve la función: las
mediciones de la vista lateral (`lat`) y de la posterior (`post`).

In [ ]:
lat, post = analizar_caso(CASO)

La misma información en una tabla. La última columna indica si los puntos usados se vieron bien (✓) o si el modelo tuvo que adivinar (⚠).

In [ ]:
tabla_mediciones(lat, post)

**Antes de seguir, revisa el dibujo:**
- ¿Los puntos caen donde deberían? Fíjate en la **cadera**, que suele quedar tapada por el
  descansabrazos o el respaldo.
- ¿Qué mediciones salieron con ⚠? Esas son las que hay que confirmar a ojo.
- La **muñeca** es la medición más débil: la mano ocupa pocos píxeles.

### 1.3 Completa la evaluación 🧩
Ahora combinamos lo que midió la computadora con lo que tú observas. Hay una tarjeta por cada paso
de la hoja RULA, con su nombre y su color. Cada tarjeta explica **qué significa** (📖) y **cómo
observarlo** (👀):

- 🤖 La computadora pudo medirlo: verás su **sugerencia** ya seleccionada.
- 🧑 La computadora no puede saberlo con una foto (fuerza, tiempo, giros): **decide tú**.

Las preguntas están en dos pestañas: **A** (brazo y muñeca) y **B** (cuello, torso y piernas).
Puedes cambiar cualquier respuesta y la hoja de evaluación del final se actualiza sola. Si
contradices a la computadora, aparecerá ✏️ para que pienses por qué.

In [ ]:
cuestionario_rula(CASO)

### 1.4 Resultado 📝
Cada paso con la respuesta elegida y sus puntos, y abajo la hoja de evaluación completa.

💡 Si cambias alguna respuesta del cuestionario, vuelve a ejecutar esta celda para actualizar el resumen.

In [ ]:
resumen(CASO)

### Para pensar
1. ¿Qué sugerencias de la computadora cambiaste? ¿Por qué?
2. El torso salió con ángulo negativo. ¿Qué significa y por qué RULA lo cuenta como 0°?
3. La cabeza parece erguida, pero el cuello puede salir con más de 20°. La computadora usa la línea
   oreja–hombro, que no es la columna cervical. ¿Qué tanto confiarías en esa medición?
4. Si cambias algún valor en `CRITERIOS` y vuelves a ejecutar el caso, ¿cambia el puntaje final?

---
## Parte 2 · Caso 2: trabajo con laptop

### 2.1 La situación
Un estudiante trabaja con la laptop sobre las piernas durante sesiones largas, en una silla
de oficina. Escribe y lee sin cambiar de postura; la laptop pesa alrededor de 1.5 kg.

In [ ]:
CASO = "02_laptop.png"

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
for eje, vista in zip(ejes, ["lateral", "posterior"]):
    eje.imshow(vistas[CASO][vista])
    eje.set_title(f"{CASOS[CASO]} · vista {vista}")
    eje.axis("off")
plt.tight_layout()
plt.show()

### 2.2 Lo que mide la computadora 🤖

In [ ]:
lat, post = analizar_caso(CASO)

In [ ]:
tabla_mediciones(lat, post)

**Antes de seguir, revisa el dibujo:**
- La cabeza está muy inclinada: ¿la oreja quedó bien ubicada?
- Observa los pies en la vista lateral: ¿están bien apoyados en el piso?
- Los brazos descansan sobre los descansabrazos o las piernas: decide si cuenta como apoyo.

### 2.3 Completa la evaluación 🧩
Recuerda usar la situación descrita arriba para las preguntas de uso muscular y fuerza/carga.

In [ ]:
cuestionario_rula(CASO)

### 2.4 Resultado 📝
Cada paso con la respuesta elegida y sus puntos, y abajo la hoja de evaluación completa.

In [ ]:
resumen(CASO)

### Para pensar
1. ¿Qué segmento aporta más puntos en este caso?
2. La laptop sobre las piernas obliga a mirar hacia abajo. ¿Qué cambiarías del puesto para bajar
   el puntaje? Prueba en el cuestionario qué pasa si el cuello y el torso quedaran neutros.
3. ¿Los brazos en los descansabrazos compensan la postura del cuello y torso?

---
## Parte 3 · Caso 3: odontóloga

### 3.1 La situación
La odontóloga realiza procedimientos de 20 a 40 minutos inclinada sobre el paciente, con
instrumentos ligeros que sostiene con precisión. Está sentada en un banco sin respaldo ni
descansabrazos.

In [ ]:
CASO = "03_odontologia.png"

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
for eje, vista in zip(ejes, ["lateral", "posterior"]):
    eje.imshow(vistas[CASO][vista])
    eje.set_title(f"{CASOS[CASO]} · vista {vista}")
    eje.axis("off")
plt.tight_layout()
plt.show()

### 3.2 Lo que mide la computadora 🤖

In [ ]:
lat, post = analizar_caso(CASO)

In [ ]:
tabla_mediciones(lat, post)

**Antes de seguir, revisa el dibujo:**
- Aquí la postura es asimétrica: compara los dos brazos en la vista posterior.
- El desnivel de hombros y la inclinación lateral pueden deberse a la postura real o a un error
  de detección. ¿Cuál es?
- Con el torso inclinado hacia adelante y el banco debajo, revisa dónde quedó la cadera.

### 3.3 Completa la evaluación 🧩
Recuerda usar la situación descrita arriba para las preguntas de uso muscular y fuerza/carga.

In [ ]:
cuestionario_rula(CASO)

### 3.4 Resultado 📝
Cada paso con la respuesta elegida y sus puntos, y abajo la hoja de evaluación completa.

In [ ]:
resumen(CASO)

### Para pensar
1. ¿Por qué este caso tiene más ajustes (pasos 1a, 9a, 10a) que los anteriores?
2. Los instrumentos pesan muy poco. ¿Por qué el puntaje sigue siendo alto?
3. En odontología la postura depende del paciente. ¿Qué modificaciones serían realistas?

---
## Parte 4 · Los tres casos lado a lado
La tabla reúne la hoja de evaluación de cada caso. Así se ve en qué grupo y en qué segmento se
concentra el riesgo de cada postura.

In [ ]:
comparar_casos()

### Para cerrar
- ¿Qué caso resultó más riesgoso? ¿Coincide con tu intuición al ver las fotos?
- ¿Qué pasos pudo resolver bien la computadora y en cuáles necesitó tu criterio?
- ¿Qué limitaciones tiene medir ángulos en una fotografía 2D? Piensa en la profundidad, los objetos
  que tapan el cuerpo y el punto desde donde se tomó la foto.

**Referencia:** McAtamney, L. y Corlett, E. N. (1993). RULA: a survey method for the investigation
of work-related upper limb disorders. *Applied Ergonomics*, 24(2), 91–99.

---
Libreta del taller **Ergonomía Digital** · Miguel Serrano Reyes, Facultad de Ingeniería, UNAM.
Código bajo licencia MIT; textos y explicaciones bajo licencia CC BY 4.0.
Repositorio: [github.com/MiguelSerranoReyes/ergonomia-digital](https://github.com/MiguelSerranoReyes/ergonomia-digital)